In [0]:
from pyspark.sql.functions import col, current_timestamp

# Define our simulation paths
landing_path = "/Volumes/workspace/default/tmp_landing/sales_data/"
checkpoint_path = "/Volumes/workspace/default/tmp_landing/_checkpoints/sales_ingest"

# Use Auto Loader to 'pick up' the file
df_ingest = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", checkpoint_path) # Tracks schema changes
  .option("header", "true")
  .load(landing_path)
  .withColumn("source_file", col("_metadata.file_path"))      # Metadata for audit
  .withColumn("ingested_at", current_timestamp())    # Metadata for audit
)

# Write to a Delta Table
query = (df_ingest.writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .outputMode("append")
  .toTable("workspace.default.bronze_sales_stream") # Change to your catalog/schema
)